In [ ]:
# 1. 首先挂载 Google Drive；结果 zip 和 manifest 最终写入该目录。
from google.colab import drive
drive.mount('/content/drive')

import json
import os
from pathlib import Path
import subprocess
import sys

DRIVE_PROJECT_ROOT = os.environ.get('SSTW_DRIVE_PROJECT_ROOT', '/content/drive/MyDrive/SSTW')
REPO_URL = os.environ.get('SSTW_REPO_URL', 'https://github.com/RICHAAARC/SSTW.git')
REPO_DIR = os.environ.get('SSTW_REPO_DIR', '/content/SSTW')
REPO_REF = os.environ.get('SSTW_REPO_REF', '').strip()
WORKFLOW_PROFILE = 'method_mechanism_validation'
os.environ['SSTW_WORKFLOW_PROFILE'] = WORKFLOW_PROFILE
print('Drive project root =', DRIVE_PROJECT_ROOT)
print(subprocess.getoutput('nvidia-smi'))


# SSTW method mechanism validation

该 Notebook 在真实 Colab GPU 上运行 16 个配对视频的最小机制验证。Notebook 只负责 Drive、代码、依赖、认证和参数编排；实际生成、机制判定和阶段包发布全部由与普通 GPU 服务器共用的 CLI 完成。该结果只证明机制执行路径可运行，不属于论文效果证据。


In [ ]:
# 2. 获取仓库代码并冻结本次执行 commit。
repo_path = Path(REPO_DIR)
if not repo_path.exists():
    subprocess.run(['git', 'clone', '--filter=blob:none', REPO_URL, REPO_DIR], check=True)
if REPO_REF:
    subprocess.run(['git', 'fetch', '--tags', 'origin'], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', '--detach', REPO_REF], cwd=REPO_DIR, check=True)
else:
    subprocess.run(['git', 'pull', '--ff-only'], cwd=REPO_DIR, check=True)
os.chdir(REPO_DIR)
RESOLVED_REPO_COMMIT = subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True
).strip()
print('SSTW repository commit =', RESOLVED_REPO_COMMIT)


In [ ]:
# 3. 安装受版本锁约束的 GPU 运行环境。
# 不使用 -U 或 Git 分支依赖，防止 Colab 与普通 GPU 服务器产生依赖漂移。
%pip install --requirement requirements/paper_runtime_lock.txt


In [ ]:
# 4. Hugging Face 认证；支持 VSCode 连接 Colab 时从 Drive 私密文件引导。
from paper_workflow.colab_utils.huggingface_authentication import bootstrap_huggingface_authentication

huggingface_authentication = bootstrap_huggingface_authentication(
    DRIVE_PROJECT_ROOT,
    environ=os.environ,
)
print('Hugging Face authentication =', huggingface_authentication)


In [ ]:
# 5. 从 Drive 私有目录加载轨迹认证密钥，不打印密钥本体。
from paper_workflow.colab_utils.trajectory_authentication import load_trajectory_authentication_from_private_drive

trajectory_authentication = load_trajectory_authentication_from_private_drive(
    DRIVE_PROJECT_ROOT,
    environ=os.environ,
)
print('trajectory authentication =', trajectory_authentication)


In [ ]:
# 6. 执行 GPU 验证；CLI 的最后阶段将结果 zip 和 manifest 打包到 Google Drive。
from workflows.streaming_command import run_streaming_command

NOTEBOOK_ROLE = 'method_mechanism_validation'
SERVER_PIPELINE = 'method_mechanism_validation'
DECISION_OUTPUT = Path('/content/sstw_method_mechanism_validation_decision.json')
server_command = [
    sys.executable,
    'scripts/run_generative_video_server_workflow.py',
    '--project-root', DRIVE_PROJECT_ROOT,
    '--repo-root', REPO_DIR,
    '--workflow-profile', WORKFLOW_PROFILE,
    '--pipeline', SERVER_PIPELINE,
    '--local-workspace-root', '/content/SSTW_stage_workspace',
    '--local-package-cache-root', '/content/SSTW_stage_packages',
    '--decision-output', str(DECISION_OUTPUT),
]
if os.environ.get('SSTW_MODEL_REVISION'):
    server_command.extend(['--model-revision', os.environ['SSTW_MODEL_REVISION']])
if os.environ.get('SSTW_INCLUDE_VIDEOS_IN_PACKAGE', 'true').lower() != 'true':
    server_command.append('--exclude-videos')

print('server workflow pipeline =', SERVER_PIPELINE)
result = run_streaming_command(server_command)
if result.returncode != 0:
    raise RuntimeError(f'method mechanism validation failed, return_code={result.returncode}')

decision = json.loads(DECISION_OUTPUT.read_text(encoding='utf-8'))
pipeline_result = decision['pipeline_results'][0]
stage_package = pipeline_result.get('stage_package', {})
if stage_package.get('stage_package_publish_status') != 'published':
    raise RuntimeError(f'Google Drive package was not published: {stage_package}')

drive_package_zip = Path(stage_package['drive_stage_package_zip'])
drive_package_manifest = Path(stage_package['stage_package_manifest_path'])
drive_root = Path(DRIVE_PROJECT_ROOT).resolve()
for package_path in (drive_package_zip, drive_package_manifest):
    if not package_path.is_file():
        raise FileNotFoundError(f'Expected Drive package is missing: {package_path}')
    package_path.resolve().relative_to(drive_root)

print('server workflow decision =', decision['server_workflow_decision'])
print('Google Drive result zip =', drive_package_zip)
print('Google Drive manifest =', drive_package_manifest)
